# Reversible PV Curve Registration Analysis

这个 Notebook 用于分析“多训练场站预训练 + 新目标站少量历史数据配准”的场景。

核心原则：

1. 公共模板只由训练场站构造，目标站不参与模板更新。
2. 容量归一化用于保留真实容量利用率。
3. 求时间映射时临时进行 P95 形状归一化，避免优化器用时间压缩拟合幅值差。
4. 时间映射限制最大偏移与局部斜率，避免出现窄尖峰或“吐出来一块”。
5. 映射函数数学上可逆；固定15分钟网格上的正反插值是近似可逆，因此需要检查 round-trip loss。

所有图片标签使用英文，避免服务器缺少中文字体。

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)


## 1. Configuration

训练站和目标站可以使用不同的校准日期范围，但最好处于相同季节。目标站只有两三周数据时，建议把源站校准范围也设置为相同月份或季节，避免把季节变化误认为场站差异。

In [ ]:
DATA_DIR = Path("/data/hjs/your_parquet_directory")
FILE_GLOB = "station=*.parquet"

TIMESTAMP_COL = "timestamp_win"
STATION_COL = "station"
POWER_HISTORY_COL = "observe_power"
HISTORY_LAST_OFFSET_MINUTES = 0

TARGET_STATION = "replace_with_target_station"
SOURCE_STATIONS = None  # None means every station except TARGET_STATION.

STATION_CAPACITY = {
    # "source_station_1": 465.0,
    # "source_station_2": 520.0,
    # "replace_with_target_station": 480.0,
}

SOURCE_MAPPING_START = pd.Timestamp("2024-01-01 00:00:00")
SOURCE_MAPPING_END = pd.Timestamp("2024-12-31 23:59:59")
TARGET_MAPPING_START = pd.Timestamp("2024-01-01 00:00:00")
TARGET_MAPPING_END = pd.Timestamp("2024-12-31 23:59:59")

POINTS_PER_HOUR = 4
POINTS_PER_DAY = 24 * POINTS_PER_HOUR
MIN_VALID_SLOTS_PER_DAY = 72
MIN_SELECTED_DAYS = 5
USE_HIGH_ENERGY_DAYS = True
HIGH_ENERGY_DAY_FRACTION = 0.40

DAYLIGHT_THRESHOLD = 0.02
N_WARP_KNOTS = 5
MAX_TIME_SHIFT_HOURS = 1.0
MIN_LOCAL_SLOPE = 0.75
MAX_LOCAL_SLOPE = 1.33
IDENTITY_PENALTY = 0.10
SMOOTHNESS_PENALTY = 0.05
GRADIENT_LOSS_WEIGHT = 0.20
N_TEMPLATE_ITERATIONS = 2
MIN_ALIGNMENT_IMPROVEMENT = 0.01


## 2. Read only mapping data

曲线配准只需要 `timestamp_win`、`station` 和 `observe_power[-1]`。未来功率和气象列不会被读取。每个 parquet 读取后立即提取最后一个历史功率值，不在内存中长期保留7天数组。

In [ ]:
def exact_last_value(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    if len(values) == 0 or not np.isfinite(values[-1]):
        return np.nan
    return float(values[-1])


def station_name(path, frame):
    values = frame[STATION_COL].dropna().astype(str).unique()
    if len(values) != 1:
        raise ValueError(f"{path.name}: station column is not constant")
    return str(values[0])


station_series = {}
files = sorted(DATA_DIR.glob(FILE_GLOB))
if not files:
    raise FileNotFoundError(f"No files matching {FILE_GLOB} under {DATA_DIR}")

for path in files:
    frame = pd.read_parquet(
        path,
        columns=[TIMESTAMP_COL, STATION_COL, POWER_HISTORY_COL],
    )
    station = station_name(path, frame)
    compact = pd.DataFrame({
        "timestamp": (
            pd.to_datetime(frame[TIMESTAMP_COL], errors="coerce")
            + pd.to_timedelta(HISTORY_LAST_OFFSET_MINUTES, unit="m")
        ),
        "power": frame[POWER_HISTORY_COL].map(exact_last_value),
    }).dropna()
    compact = (
        compact.groupby("timestamp", as_index=False)["power"]
        .median()
        .sort_values("timestamp")
    )
    compact["station"] = station
    station_series[station] = compact
    del frame

if TARGET_STATION not in station_series:
    raise KeyError(f"TARGET_STATION={TARGET_STATION!r} is missing")

if SOURCE_STATIONS is None:
    source_stations = [s for s in station_series if s != TARGET_STATION]
else:
    source_stations = [str(s) for s in SOURCE_STATIONS]

if not source_stations:
    raise ValueError("At least one source station is required")
if TARGET_STATION in source_stations:
    raise ValueError("TARGET_STATION must not be a source station")

required_stations = source_stations + [TARGET_STATION]
missing_capacity = [s for s in required_stations if s not in STATION_CAPACITY]
if missing_capacity:
    raise KeyError(f"STATION_CAPACITY is missing: {missing_capacity}")

summary = pd.DataFrame([
    {
        "station": station,
        "role": "target" if station == TARGET_STATION else "source",
        "rows": len(station_series[station]),
        "start": station_series[station]["timestamp"].min(),
        "end": station_series[station]["timestamp"].max(),
        "capacity": float(STATION_CAPACITY[station]),
    }
    for station in required_stations
])
display(summary)


## 3. Daily curves and representative curves

先重建每天96点容量归一化曲线。可选地选择日发电量较高的日期，再计算代表性中位数曲线。这里的中位数仅用于估计映射，本身不可逆；实际曲线的时间变换仍可近似逆向恢复。

In [ ]:
slot_grid = np.arange(POINTS_PER_DAY)
grid01 = slot_grid.astype(np.float64) / POINTS_PER_DAY
hour_grid = slot_grid / POINTS_PER_HOUR
canonical_knots = np.linspace(0.0, 1.0, N_WARP_KNOTS)


def date_range_for_station(station):
    if station == TARGET_STATION:
        return TARGET_MAPPING_START, TARGET_MAPPING_END
    return SOURCE_MAPPING_START, SOURCE_MAPPING_END


def build_daily_matrix(frame, station):
    start, end = date_range_for_station(station)
    data = frame[frame["timestamp"].between(start, end)].copy()
    if data.empty:
        raise ValueError(f"No mapping data for station={station}")

    data["date"] = data["timestamp"].dt.date
    data["slot"] = (
        data["timestamp"].dt.hour * POINTS_PER_HOUR
        + data["timestamp"].dt.minute // 15
    )
    data["power_ratio"] = (
        data["power"] / float(STATION_CAPACITY[station])
    )

    daily = data.pivot_table(
        index="date",
        columns="slot",
        values="power_ratio",
        aggfunc="median",
    ).reindex(columns=slot_grid)

    valid_count = daily.notna().sum(axis=1)
    daily = daily[valid_count >= MIN_VALID_SLOTS_PER_DAY]
    daily = daily.interpolate(axis=1, limit_direction="both")
    daily = daily.dropna()
    if len(daily) < MIN_SELECTED_DAYS:
        raise ValueError(
            f"station={station} has only {len(daily)} valid days"
        )
    return daily


def select_days(daily):
    if not USE_HIGH_ENERGY_DAYS:
        return daily
    daily_energy = daily.clip(lower=0.0).sum(axis=1)
    n_select = max(
        MIN_SELECTED_DAYS,
        int(math.ceil(len(daily) * HIGH_ENERGY_DAY_FRACTION)),
    )
    n_select = min(n_select, len(daily))
    selected_index = daily_energy.nlargest(n_select).index
    return daily.loc[selected_index].sort_index()


daily_curves = {}
selected_daily_curves = {}
capacity_curves = {}
selection_rows = []
for station in required_stations:
    daily = build_daily_matrix(station_series[station], station)
    selected = select_days(daily)
    daily_curves[station] = daily
    selected_daily_curves[station] = selected
    capacity_curves[station] = selected.median(axis=0).to_numpy(dtype=float)
    selection_rows.append({
        "station": station,
        "valid_days": len(daily),
        "selected_days": len(selected),
        "representative_peak": float(np.max(capacity_curves[station])),
    })

capacity_curves_df = pd.DataFrame(capacity_curves, index=slot_grid)
display(pd.DataFrame(selection_rows))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for station in source_stations:
    ax.plot(
        hour_grid, capacity_curves_df[station],
        color="tab:orange", alpha=0.35, lw=1.4, label=None,
    )
ax.plot(
    hour_grid, capacity_curves_df[TARGET_STATION],
    color="tab:green", lw=2.8, label=f"Target: {TARGET_STATION}",
)
ax.plot([], [], color="tab:orange", lw=2, label="Source stations")
ax.set(
    title="Capacity-Normalized Representative Curves Before Registration",
    xlabel="Physical Time (hour)",
    ylabel="Power / Installed Capacity",
    xlim=(0, 23.75),
)
ax.legend()
plt.tight_layout()
plt.show()


## 4. Amplitude-invariant shapes

P95形状归一化只用于估计时间映射。实际TabM数据不应进行逐曲线P95归一化。这里同时打印每个站的形状尺度，帮助判断容量归一化后仍有多大的稳定幅值差异。

In [ ]:
def normalize_shape(curve):
    curve = np.clip(np.asarray(curve, dtype=float), 0.0, None)
    daylight = curve > DAYLIGHT_THRESHOLD
    if daylight.sum() < 8:
        raise ValueError("Too few daylight points for shape normalization")
    scale = float(np.quantile(curve[daylight], 0.95))
    if not np.isfinite(scale) or scale <= 1e-8:
        raise ValueError(f"Invalid shape scale: {scale}")
    return curve / scale, scale


shape_curves = {}
shape_scales = {}
for station in required_stations:
    shape_curves[station], shape_scales[station] = normalize_shape(
        capacity_curves[station]
    )

shape_curves_df = pd.DataFrame(shape_curves, index=slot_grid)
display(pd.DataFrame({
    "station": required_stations,
    "role": ["target" if s == TARGET_STATION else "source" for s in required_stations],
    "shape_scale": [shape_scales[s] for s in required_stations],
}))

fig, ax = plt.subplots(figsize=(12, 5))
for station in source_stations:
    ax.plot(hour_grid, shape_curves_df[station], color="tab:orange", alpha=0.35, lw=1.4)
ax.plot(hour_grid, shape_curves_df[TARGET_STATION], color="tab:green", lw=2.8, label=f"Target: {TARGET_STATION}")
ax.plot([], [], color="tab:orange", lw=2, label="Source stations")
ax.set(
    title="Amplitude-Invariant Shapes Before Registration",
    xlabel="Physical Time (hour)",
    ylabel="Representative Curve / Curve P95",
    xlim=(0, 23.75),
)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Constrained monotonic registration

公共模板只由源站建立。目标站在模板固定后单独拟合。优化同时限制控制点最大偏移和局部斜率；如果形状误差改善不足，则自动使用恒等映射。

In [ ]:
MAX_SHIFT_FRACTION = MAX_TIME_SHIFT_HOURS / 24.0
IDENTITY_KNOTS = canonical_knots.copy()


def apply_warp(curve, source_knots):
    source_position = np.interp(grid01, canonical_knots, source_knots)
    registered = np.interp(
        source_position,
        np.r_[grid01, 1.0],
        np.r_[curve, curve[0]],
    )
    return registered, source_position


def inverse_warp(registered_curve, source_position):
    if not np.all(np.diff(source_position) > 0):
        raise ValueError("source_position must be strictly increasing")
    canonical_position = np.interp(
        grid01,
        np.r_[source_position, 1.0],
        np.r_[grid01, 1.0],
    )
    restored = np.interp(
        canonical_position,
        np.r_[grid01, 1.0],
        np.r_[registered_curve, registered_curve[0]],
    )
    return restored, canonical_position


def alignment_rmse(curve, template):
    daylight = (curve > DAYLIGHT_THRESHOLD) | (template > DAYLIGHT_THRESHOLD)
    if daylight.sum() < 8:
        daylight = np.ones_like(template, dtype=bool)
    return float(np.sqrt(np.mean((curve[daylight] - template[daylight]) ** 2)))


def fit_constrained_warp(curve, template):
    x0 = canonical_knots[1:-1].copy()

    def unpack(x):
        return np.r_[0.0, x, 1.0]

    def local_slopes(x):
        knots = unpack(x)
        return np.diff(knots) / np.diff(canonical_knots)

    bounds = [
        (
            max(0.0, knot - MAX_SHIFT_FRACTION),
            min(1.0, knot + MAX_SHIFT_FRACTION),
        )
        for knot in canonical_knots[1:-1]
    ]

    def objective(x):
        knots = unpack(x)
        registered, _ = apply_warp(curve, knots)
        daylight = (template > DAYLIGHT_THRESHOLD) | (registered > DAYLIGHT_THRESHOLD)
        if daylight.sum() < 8:
            daylight = np.ones_like(template, dtype=bool)

        value_loss = np.mean((registered[daylight] - template[daylight]) ** 2)
        registered_gradient = np.gradient(registered)
        template_gradient = np.gradient(template)
        gradient_loss = np.mean(
            (registered_gradient[daylight] - template_gradient[daylight]) ** 2
        )
        identity_loss = np.mean((knots - canonical_knots) ** 2)
        smoothness_loss = np.mean(np.diff(knots, n=2) ** 2)
        return float(
            value_loss
            + GRADIENT_LOSS_WEIGHT * gradient_loss
            + IDENTITY_PENALTY * identity_loss
            + SMOOTHNESS_PENALTY * smoothness_loss
        )

    result = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints=[
            {"type": "ineq", "fun": lambda x: local_slopes(x) - MIN_LOCAL_SLOPE},
            {"type": "ineq", "fun": lambda x: MAX_LOCAL_SLOPE - local_slopes(x)},
        ],
        options={"maxiter": 800, "ftol": 1e-11, "disp": False},
    )

    fitted_knots = unpack(result.x) if result.success else IDENTITY_KNOTS.copy()
    registered, position = apply_warp(curve, fitted_knots)
    before = alignment_rmse(curve, template)
    after = alignment_rmse(registered, template)
    improvement = (before - after) / before if before > 0 else 0.0

    accepted = bool(result.success and improvement >= MIN_ALIGNMENT_IMPROVEMENT)
    if not accepted:
        fitted_knots = IDENTITY_KNOTS.copy()
        registered, position = apply_warp(curve, fitted_knots)
        after = alignment_rmse(registered, template)
        improvement = (before - after) / before if before > 0 else 0.0

    return {
        "knots": fitted_knots,
        "registered": registered,
        "position": position,
        "before": before,
        "after": after,
        "improvement": improvement,
        "accepted": accepted,
        "optimizer_success": bool(result.success),
        "optimizer_message": str(result.message),
    }


In [ ]:
# Build a source-only common template. The target never updates this template.
template = np.median(
    np.stack([shape_curves[s] for s in source_stations]),
    axis=0,
)

for iteration in range(N_TEMPLATE_ITERATIONS):
    aligned_sources = []
    for station in source_stations:
        aligned_sources.append(
            fit_constrained_warp(shape_curves[station], template)["registered"]
        )
    new_template = np.median(np.stack(aligned_sources), axis=0)
    change = float(np.sqrt(np.mean((new_template - template) ** 2)))
    print(
        f"Source-template iteration {iteration + 1}/{N_TEMPLATE_ITERATIONS}: "
        f"RMSE change={change:.8f}"
    )
    template = new_template

registration = {}
for station in required_stations:
    registration[station] = fit_constrained_warp(shape_curves[station], template)

# Apply the shape-derived warp to the original capacity-normalized representative curves.
registered_capacity = {}
restored_capacity = {}
inverse_positions = {}
for station in required_stations:
    registered_capacity[station], _ = apply_warp(
        capacity_curves[station], registration[station]["knots"]
    )
    restored_capacity[station], inverse_positions[station] = inverse_warp(
        registered_capacity[station], registration[station]["position"]
    )

registered_shape_df = pd.DataFrame(
    {s: registration[s]["registered"] for s in required_stations},
    index=slot_grid,
)
registered_capacity_df = pd.DataFrame(registered_capacity, index=slot_grid)
restored_capacity_df = pd.DataFrame(restored_capacity, index=slot_grid)


## 6. Registration diagnostics

重点关注 `max_shift_min`、`min_local_slope`、`max_local_slope` 和 `roundtrip_rmse_capacity`。配准误差下降不代表预测一定改善，但异常时间偏移或斜率意味着映射不应使用。

In [ ]:
metric_rows = []
for station in required_stations:
    result = registration[station]
    slopes = np.diff(result["knots"]) / np.diff(canonical_knots)
    original = capacity_curves[station]
    restored = restored_capacity[station]
    error = restored - original
    metric_rows.append({
        "station": station,
        "role": "target" if station == TARGET_STATION else "source",
        "accepted": result["accepted"],
        "shape_rmse_before": result["before"],
        "shape_rmse_after": result["after"],
        "shape_improvement_pct": 100.0 * result["improvement"],
        "max_shift_min": float(np.max(np.abs(result["knots"] - canonical_knots)) * 24 * 60),
        "min_local_slope": float(slopes.min()),
        "max_local_slope": float(slopes.max()),
        "roundtrip_mse_capacity": float(np.mean(error ** 2)),
        "roundtrip_rmse_capacity": float(np.sqrt(np.mean(error ** 2))),
        "roundtrip_max_abs_capacity": float(np.max(np.abs(error))),
        "roundtrip_rmse_power": float(
            np.sqrt(np.mean(error ** 2)) * STATION_CAPACITY[station]
        ),
    })

metrics = pd.DataFrame(metric_rows).sort_values(["role", "station"]).reset_index(drop=True)
display(metrics.style.format({
    "shape_rmse_before": "{:.6f}",
    "shape_rmse_after": "{:.6f}",
    "shape_improvement_pct": "{:.2f}%",
    "max_shift_min": "{:.2f}",
    "min_local_slope": "{:.4f}",
    "max_local_slope": "{:.4f}",
    "roundtrip_mse_capacity": "{:.10f}",
    "roundtrip_rmse_capacity": "{:.8f}",
    "roundtrip_max_abs_capacity": "{:.8f}",
    "roundtrip_rmse_power": "{:.6f}",
}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for station in source_stations:
    axes[0].plot(hour_grid, shape_curves_df[station], color="tab:orange", alpha=0.30, lw=1.3)
    axes[1].plot(hour_grid, registered_shape_df[station], color="tab:orange", alpha=0.30, lw=1.3)
axes[0].plot(hour_grid, shape_curves_df[TARGET_STATION], color="tab:green", lw=2.6, label="Target")
axes[1].plot(hour_grid, registered_shape_df[TARGET_STATION], color="tab:green", lw=2.6, label="Target")
axes[0].plot(hour_grid, template, "k--", lw=2.2, label="Source-only template")
axes[1].plot(hour_grid, template, "k--", lw=2.2, label="Source-only template")
axes[0].set_title("Shape Curves Before Registration")
axes[1].set_title("Shape Curves After Constrained Registration")
for ax in axes:
    ax.set(xlabel="Time (hour)", xlim=(0, 23.75))
axes[0].set_ylabel("P95-Normalized Shape")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for station in source_stations:
    axes[0].plot(hour_grid, capacity_curves_df[station], color="tab:orange", alpha=0.30, lw=1.3)
    axes[1].plot(hour_grid, registered_capacity_df[station], color="tab:orange", alpha=0.30, lw=1.3)
axes[0].plot(hour_grid, capacity_curves_df[TARGET_STATION], color="tab:green", lw=2.6, label="Target")
axes[1].plot(hour_grid, registered_capacity_df[TARGET_STATION], color="tab:green", lw=2.6, label="Target")
axes[0].set_title("Capacity-Normalized Curves Before Registration")
axes[1].set_title("Capacity-Normalized Curves After Shape-Derived Warp")
for ax in axes:
    ax.set(xlabel="Time (hour)", xlim=(0, 23.75))
axes[0].set_ylabel("Power / Installed Capacity")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].plot(hour_grid, hour_grid, "k--", lw=1.8, label="Identity")
for station in source_stations:
    axes[0].plot(
        hour_grid, registration[station]["position"] * 24.0,
        color="tab:orange", alpha=0.35, lw=1.3,
    )
axes[0].plot(
    hour_grid, registration[TARGET_STATION]["position"] * 24.0,
    color="tab:green", lw=2.6, label="Target",
)
axes[0].set(
    title="Monotonic Time Mappings",
    xlabel="Canonical Time tau (hour)",
    ylabel="Source Physical Time psi(tau) (hour)",
    xlim=(0, 23.75), ylim=(0, 23.75),
)
axes[0].legend()

for station in source_stations:
    slopes = np.diff(registration[station]["knots"]) / np.diff(canonical_knots)
    axes[1].step(
        canonical_knots[:-1] * 24.0, slopes, where="post",
        color="tab:orange", alpha=0.35, lw=1.3,
    )
target_slopes = (
    np.diff(registration[TARGET_STATION]["knots"])
    / np.diff(canonical_knots)
)
axes[1].step(
    canonical_knots[:-1] * 24.0, target_slopes, where="post",
    color="tab:green", lw=2.6, label="Target",
)
axes[1].axhline(1.0, color="black", ls="--", lw=1.5, label="Identity slope")
axes[1].axhline(MIN_LOCAL_SLOPE, color="tab:red", ls=":", lw=1.5)
axes[1].axhline(MAX_LOCAL_SLOPE, color="tab:red", ls=":", lw=1.5)
axes[1].set(
    title="Local Warp Slopes",
    xlabel="Canonical Knot Time (hour)",
    ylabel="Delta Source Time / Delta Canonical Time",
)
axes[1].legend()
plt.tight_layout()
plt.show()


## 7. Inspect forward and inverse registration

修改 `STATION_TO_INSPECT`。左图比较原始、配准和逆向恢复代表曲线；右图只放大原始与恢复曲线。恢复误差小表示离散插值损失较小，但仍需同时检查映射斜率是否具有物理合理性。

In [ ]:
STATION_TO_INSPECT = TARGET_STATION

if STATION_TO_INSPECT not in required_stations:
    raise KeyError(f"Unknown station: {STATION_TO_INSPECT}")

original = capacity_curves_df[STATION_TO_INSPECT].to_numpy()
registered = registered_capacity_df[STATION_TO_INSPECT].to_numpy()
restored = restored_capacity_df[STATION_TO_INSPECT].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(hour_grid, original, lw=2.3, label="Original")
axes[0].plot(hour_grid, registered, lw=2.0, label="Registered")
axes[0].plot(hour_grid, restored, "--", lw=2.0, label="Round-trip restored")
axes[0].set(
    title=f"Forward and Inverse Registration: {STATION_TO_INSPECT}",
    xlabel="Time (hour)", ylabel="Power / Installed Capacity",
)
axes[0].legend()

axes[1].plot(hour_grid, original, lw=2.5, label="Original")
axes[1].plot(hour_grid, restored, "--", lw=2.2, label="Restored")
axes[1].fill_between(
    hour_grid, original, restored, color="tab:red", alpha=0.18, label="Interpolation error",
)
axes[1].set(
    title="Round-trip Recovery Error",
    xlabel="Physical Time (hour)", ylabel="Power / Installed Capacity",
)
axes[1].legend()
plt.tight_layout()
plt.show()

error = restored - original
print(f"station: {STATION_TO_INSPECT}")
print(f"round-trip MSE:  {np.mean(error ** 2):.10f}")
print(f"round-trip RMSE: {np.sqrt(np.mean(error ** 2)):.8f}")
print(f"round-trip MAE:  {np.mean(np.abs(error)):.8f}")
print(f"max abs error:   {np.max(np.abs(error)):.8f}")


## Interpretation

- `Capacity-Normalized Curves` 仍可保留0.55与0.8的幅值差，这是正常的容量利用率差异。
- `Amplitude-Invariant Shapes` 用于判断曲线是否仅存在时间相位差。
- 配准后的形状曲线应当平滑收拢，不应出现窄而高的突出块。
- `max_shift_min` 接近上限，说明该站正在使用全部允许的时间偏移。
- 局部斜率达到上下限，说明优化器仍有强烈的压缩或拉伸倾向，应考虑降低最大偏移、减少控制点或直接使用恒等映射。
- `roundtrip_rmse` 很小只说明插值近似可逆，不证明配准具有正确物理意义。
- 如果形状归一化后各站本来已经很接近，最优结果应接近恒等映射；不要为了降低对齐误差而强制扭曲。
- 目标站只参与最后的 `fit_constrained_warp(target_shape, fixed_source_template)`，绝不能重新计算源站模板。